In [ ]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()


# ============================================================
# DENSE LAYER — WITH L1 + L2 REGULARIZATION
# ============================================================

class Layer_Dense:
    def __init__(self, n_inputs, n_neurons,
                 weight_regularizer_l1=0, weight_regularizer_l2=0,
                 bias_regularizer_l1=0,   bias_regularizer_l2=0):

        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases  = np.zeros((1, n_neurons))

        # store regularization strengths — 0 means disabled
        self.weight_regularizer_l1 = weight_regularizer_l1
        self.weight_regularizer_l2 = weight_regularizer_l2
        self.bias_regularizer_l1   = bias_regularizer_l1
        self.bias_regularizer_l2   = bias_regularizer_l2

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        # base gradients — same as before
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases  = np.sum(dvalues, axis=0, keepdims=True)

        # ── L1 on weights ─────────────────────────────────
        if self.weight_regularizer_l1 > 0:
            dL1 = np.ones_like(self.weights)   # +1 for positive weights
            dL1[self.weights < 0] = -1          # -1 for negative weights
            # add L1 gradient to weight gradient
            self.dweights += self.weight_regularizer_l1 * dL1

        # ── L2 on weights ─────────────────────────────────
        if self.weight_regularizer_l2 > 0:
            # gradient of w² = 2w → proportional push toward zero
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        # ── L1 on biases ──────────────────────────────────
        if self.bias_regularizer_l1 > 0:
            dL1 = np.ones_like(self.biases)
            dL1[self.biases < 0] = -1
            self.dbiases += self.bias_regularizer_l1 * dL1

        # ── L2 on biases ──────────────────────────────────
        if self.bias_regularizer_l2 > 0:
            self.dbiases += 2 * self.bias_regularizer_l2 * self.biases

        # gradient on inputs — pass to previous layer
        self.dinputs = np.dot(dvalues, self.weights.T)


# ============================================================
# RELU
# ============================================================

class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


# ============================================================
# SOFTMAX
# ============================================================

class Activation_Softmax:
    def forward(self, inputs):
        self.inputs   = inputs
        exp_values    = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output   = probabilities

    def predictions(self, outputs):
        return np.argmax(outputs, axis=1)


# ============================================================
# LOSS BASE CLASS — NOW CALCULATES REGULARIZATION LOSS TOO
# ============================================================

class Loss:

    def regularization_loss(self):
        reg_loss = 0  # start at zero

        # loop through all trainable layers
        for layer in self.trainable_layers:

            # L1 weights — sum of absolute values
            if layer.weight_regularizer_l1 > 0:
                reg_loss += layer.weight_regularizer_l1 * \
                            np.sum(np.abs(layer.weights))

            # L2 weights — sum of squared values
            if layer.weight_regularizer_l2 > 0:
                reg_loss += layer.weight_regularizer_l2 * \
                            np.sum(layer.weights * layer.weights)

            # L1 biases
            if layer.bias_regularizer_l1 > 0:
                reg_loss += layer.bias_regularizer_l1 * \
                            np.sum(np.abs(layer.biases))

            # L2 biases
            if layer.bias_regularizer_l2 > 0:
                reg_loss += layer.bias_regularizer_l2 * \
                            np.sum(layer.biases * layer.biases)

        return reg_loss

    def remember_trainable_layers(self, trainable_layers):
        # loss needs to know which layers have weights
        # so it can calculate regularization loss for them
        self.trainable_layers = trainable_layers

    def calculate(self, output, y, *, include_regularization=False):
        sample_losses = self.forward(output, y)
        data_loss     = np.mean(sample_losses)

        if not include_regularization:
            return data_loss

        # return both separately so we can track them
        return data_loss, self.regularization_loss()


# ============================================================
# CATEGORICAL CROSS ENTROPY
# ============================================================

class Loss_CategoricalCrossentropy(Loss):

    def forward(self, y_pred, y_true):
        samples        = len(y_pred)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        if len(y_true.shape) == 1:       # sparse
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:     # one-hot
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        return -np.log(correct_confidences)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels  = len(dvalues[0])

        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        self.dinputs = -y_true / dvalues
        self.dinputs = self.dinputs / samples


# ============================================================
# COMBINED SOFTMAX + LOSS
# ============================================================

class Activation_Softmax_Loss_CategoricalCrossentropy:

    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss       = Loss_CategoricalCrossentropy()

    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.output = self.activation.output
        return self.loss.calculate(self.output, y_true)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)

        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs / samples


# ============================================================
# ADAM OPTIMIZER
# ============================================================

class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0.0,
                 epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate         = learning_rate
        self.current_learning_rate = learning_rate
        self.decay                 = decay
        self.iterations            = 0
        self.epsilon               = epsilon
        self.beta_1                = beta_1
        self.beta_2                = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * \
                (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache     = np.zeros_like(layer.weights)
            layer.bias_momentums   = np.zeros_like(layer.biases)
            layer.bias_cache       = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta_1 * layer.weight_momentums + \
                                 (1 - self.beta_1) * layer.dweights
        layer.bias_momentums   = self.beta_1 * layer.bias_momentums   + \
                                 (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / \
            (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected   = layer.bias_momentums   / \
            (1 - self.beta_1 ** (self.iterations + 1))

        layer.weight_cache = self.beta_2 * layer.weight_cache + \
                             (1 - self.beta_2) * layer.dweights ** 2
        layer.bias_cache   = self.beta_2 * layer.bias_cache   + \
                             (1 - self.beta_2) * layer.dbiases  ** 2

        weight_cache_corrected = layer.weight_cache / \
            (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected   = layer.bias_cache   / \
            (1 - self.beta_2 ** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * \
                          weight_momentums_corrected / \
                         (np.sqrt(weight_cache_corrected) + self.epsilon)
        layer.biases  += -self.current_learning_rate * \
                          bias_momentums_corrected   / \
                         (np.sqrt(bias_cache_corrected)   + self.epsilon)

    def post_update_params(self):
        self.iterations += 1


# ============================================================
# FULL TRAINING LOOP WITH REGULARIZATION
# ============================================================

X, y = spiral_data(samples=100, classes=3)

# L2 regularization on dense1 — common practice on hidden layers
# 5e-4 = 0.0005 — small enough to not hurt learning, big enough to regularize
dense1 = Layer_Dense(2, 64,
                     weight_regularizer_l2=5e-4,
                     bias_regularizer_l2=5e-4)
relu1          = Activation_ReLU()
dense2         = Layer_Dense(64, 3)   # output layer usually not regularized
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()
optimizer      = Optimizer_Adam(learning_rate=0.05, decay=5e-7)

# tell loss which layers are trainable → for regularization calculation
loss_activation.loss.remember_trainable_layers([dense1, dense2])

for epoch in range(10001):

    # ── FORWARD ──────────────────────────────────────────
    dense1.forward(X)
    relu1.forward(dense1.output)
    dense2.forward(relu1.output)

    # data loss only first
    data_loss = loss_activation.forward(dense2.output, y)

    # add regularization loss on top
    reg_loss   = loss_activation.loss.regularization_loss()
    total_loss = data_loss + reg_loss   # this is what we actually minimize

    predictions = np.argmax(loss_activation.output, axis=1)
    accuracy    = np.mean(predictions == y)

    if not epoch % 1000:
        print(f'epoch: {epoch:5d} | '
              f'acc: {accuracy:.4f} | '
              f'loss: {total_loss:.4f} | '
              f'data_loss: {data_loss:.4f} | '
              f'reg_loss: {reg_loss:.4f} | '
              f'lr: {optimizer.current_learning_rate:.6f}')

    # ── BACKWARD ─────────────────────────────────────────
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    relu1.backward(dense2.dinputs)
    dense1.backward(relu1.dinputs)   # L1/L2 gradients added inside here

    # ── UPDATE ───────────────────────────────────────────
    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()